In [1]:
print("hello")

hello


In [14]:
##RAG PIPELINE-DATA INGESTION TO VECTOR DB PIP
import os
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\kisho\AppData\Local\Temp\ipykernel_30388\3911921199.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader


In [15]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

### Read all PDF inside directory

def process_all_pdf(pdf_directory):

    all_documents = []

    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:

        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal Documents loaded: {len(all_documents)}")

    return all_documents


# Process all PDFs in directory
all_pdf_documents = process_all_pdf(
    r"C:\Users\kisho\OneDrive\Desktop\AIML LEARN\data\pdf"
)

Found 4 PDF files to process

Processing: DEEP LEARNING ( R20A06610 ).pdf
Loaded 170 pages

Processing: DL UNIT 1 notes_Updated (1).pdf
Loaded 30 pages

Processing: MACHINE LEARNING(R17A0534).pdf
Loaded 120 pages

Processing: Neural-Networks.pdf
Loaded 60 pages

Total Documents loaded: 380


In [16]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-08-09T10:02:17+00:00', 'moddate': '2024-08-09T15:33:26+05:30', 'source': 'C:\\Users\\kisho\\OneDrive\\Desktop\\AIML LEARN\\data\\pdf\\DEEP LEARNING ( R20A06610 ).pdf', 'total_pages': 170, 'page': 0, 'page_label': '1', 'source_file': 'DEEP LEARNING ( R20A06610 ).pdf', 'file_type': 'pdf'}, page_content='DEPARTMENT OF  \n          COMPUTER SCIENCE AND ENGINEERING \n                              DIGITAL NOTES \nON                                                \nDEEP LEARNING \n(R20A6610) \n  \n  \n                                      Prepared by \n                                      K.Chandusha \n \n \n    MALLA REDDY COLLEGE OF                             \nENGINEERING&TECHNOLOGY (AutonomousInstitution–UGC,Govt.ofIndia) \nRecognizedunder2(f)and12(B)ofUGCACT1956 \n(AffiliatedtoJNTUH,Hyderabad,Approved byAICTE-AccreditedbyNBA&NAAC–‘A’Grade-ISO9001:20

In [17]:
##Text splitting->chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)}documents into {len(split_docs)}chunks")

    #shwo example
    if split_docs:
        print(f"\nExample chunk")
        print(f"Content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")
    return split_docs



In [18]:
chunks=split_documents(all_pdf_documents)

Split 380documents into 889chunks

Example chunk
Content:DEPARTMENT OF  
          COMPUTER SCIENCE AND ENGINEERING 
                              DIGITAL NOTES 
ON                                                
DEEP LEARNING 
(R20A6610) 
  
  
           ...
Metadata:{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-08-09T10:02:17+00:00', 'moddate': '2024-08-09T15:33:26+05:30', 'source': 'C:\\Users\\kisho\\OneDrive\\Desktop\\AIML LEARN\\data\\pdf\\DEEP LEARNING ( R20A06610 ).pdf', 'total_pages': 170, 'page': 0, 'page_label': '1', 'source_file': 'DEEP LEARNING ( R20A06610 ).pdf', 'file_type': 'pdf'}


Embeddings and vector store db

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer  #embedding model
import chromadb  
from chromadb.config import Settings
import uuid  #id for data in db
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity  # retriveal from vectordb

In [7]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-miniLM-L6-v2"): ##vector conversion open source in huggingface
        '''Initialize the embedding manager
        Args:
        model_name:Hugging Face model name for senetnce embedding'''
        self.model_name=model_name
        self.model=None
        self._load_model()


    def _load_model(self): #protected function
        '''Load senetencetransformer model'''
        try:
            print(f"Loading Embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully.Embedding dimension:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading model{self.model_name}:{e}")
            raise


    def generate_embeddings(self,texts:List[str])->np.ndarray:
                ''''Generate embeddings fro list of texts
                Args:
                texts:List of text strings to Embed
                Returns:
                numpy array of embeddings with shape(len(texts),embedding_dim)
                '''
                if not self.model:
                    raise ValueError("Model not loaded")
                print(f"Generating embeddings for{len(texts)}texts")
                embeddings=self.model.encode(texts,show_progress_bar=True)
                print(f"Genrated embeddings with shape:{embeddings.shape}")
                return embeddings

    
##initialize embedding manager
embedding_manager=EmbeddingManager()
embedding_manager



Loading Embedding model:all-miniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\kisho\OneDrive\Desktop\AIML LEARN\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kisho\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded successfully.Embedding dimension:384


C:\Users\kisho\AppData\Local\Temp\ipykernel_30388\1424610352.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded successfully.Embedding dimension:{self.model.get_sentence_embedding_dimension()}")


VectorStore

In [21]:
import os
class VectorStore:
    """Manage document embeddings in ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistent ChromaDB directory
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector Store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to vector store.
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents to vector store"
        )

        # Prepare data for ChromaDB
        ids = []
        metadata_list = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadata_list.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to ChromaDB
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadata_list,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise


# Initialize vector store
vector_store = VectorStore()

vector_store

Vector Store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [19]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-08-09T10:02:17+00:00', 'moddate': '2024-08-09T15:33:26+05:30', 'source': 'C:\\Users\\kisho\\OneDrive\\Desktop\\AIML LEARN\\data\\pdf\\DEEP LEARNING ( R20A06610 ).pdf', 'total_pages': 170, 'page': 0, 'page_label': '1', 'source_file': 'DEEP LEARNING ( R20A06610 ).pdf', 'file_type': 'pdf'}, page_content='DEPARTMENT OF  \n          COMPUTER SCIENCE AND ENGINEERING \n                              DIGITAL NOTES \nON                                                \nDEEP LEARNING \n(R20A6610) \n  \n  \n                                      Prepared by \n                                      K.Chandusha \n \n \n    MALLA REDDY COLLEGE OF                             \nENGINEERING&TECHNOLOGY (AutonomousInstitution–UGC,Govt.ofIndia) \nRecognizedunder2(f)and12(B)ofUGCACT1956 \n(AffiliatedtoJNTUH,Hyderabad,Approved byAICTE-AccreditedbyNBA&NAAC–‘A’Grade-ISO9001:20

In [22]:
##convert teh text to embeddings
texts=[doc.page_content for doc in chunks ]
texts

##genrtae embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store in the vectord
vector_store.add_documents(chunks,embeddings)


Generating embeddings for889texts


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Genrated embeddings with shape:(889, 384)
Adding 889 documents to vector store
Successfully added 889 documents to vector store
Total documents in collection: 889


retriever pipeline from vectorstore

user query also converted into embeddings for retrieval

In [58]:
class RAGRetriever:
    """Handles query-based retrieval from vector store"""

    def __init__(
        self,
        vectorStore,
        embedding_manager: EmbeddingManager
    ):
        self.vector_store = vectorStore
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        print(f"Retrieving Documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        try:

            # Search ChromaDB
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # Convert distance to similarity
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,

                            # IMPORTANT: individual metadata
                            "metadata": metadata,

                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents (after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Initialize retriever
rag_retriever = RAGRetriever(
    vectorStore,
    embedding_manager
)

In [59]:
rag_retriever

In [60]:
rag_retriever.retrieve("Typesofneuralnetworks")

Retrieving Documents for query: 'Typesofneuralnetworks'
Top k: 5, Score threshold: 0.0
Generating embeddings for1texts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Genrated embeddings with shape:(1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_dccdbd67_39',
  'content': 'B.Tech–CSE R-20 \nDeepLearning \n \n \n \n \n \n \n \nHowdoArtificialNeuralNetworkslearn? \n \nArtificial neural networks are trained using a training set. For example, suppose you  \nwant to teach an ANN to recognize a cat. Then it is shown thousands of different images of \ncatssothatthenetworkcanlearntoidentifyacat.Oncetheneuralnetworkhasbeen trained enough \nusing images of cats, then you need to check if it can identify cat images cor rectly. This is \ndone by making the ANN classify the images it is provided by deciding whether they are cat \nimages or not.The output obtained by the ANN is corroborated by a human -provided \ndescription of whether the image is a cat image or not. \nIf the ANN identifies incorrectly then back-propagation is used to adjust whatever it \nhas learned during training. Backpropagationis done by fine -tuning the weights of the \nconnections in ANN units based on the error rate obtained.  This process continues un

Integration VectorDB Context pipeline with LLM output


In [69]:
## Simple RAG pipeline with Gemini LLM

from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0.1
)


# Simple RAG function:
# Retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):

    # Retrieve relevant documents
    results = retriever.retrieve(
        query,
        top_k=top_k
    )

    # Combine retrieved document contents
    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""

    # If no relevant documents were found
    if not context:
        return "No relevant context found"

    # Create prompt
    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    # Send prompt to Gemini
    response = llm.invoke(prompt)

    return response.content

In [70]:
answer=rag_simple("What is Deep Learning",rag_retriever,llm)
answer


Retrieving Documents for query: 'What is Deep Learning'
Top k: 3, Score threshold: 0.0
Generating embeddings for1texts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Genrated embeddings with shape:(1, 384)
Retrieved 3 documents (after filtering)


'Deep Learning is a subfield of Machine Learning that involves the use of deep neural networks to model and solve complex problems by learning from large amounts of data, automatically discovering hierarchical patterns and features, and achieving significant success in fields like image recognition, natural language processing, and speech recognition.'

In [71]:
results = rag_retriever.retrieve(
    "What is Deep Learning?",
    top_k=3
)

print(type(results[0]["metadata"]))
print(results[0]["metadata"])

Retrieving Documents for query: 'What is Deep Learning?'
Top k: 3, Score threshold: 0.0
Generating embeddings for1texts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Genrated embeddings with shape:(1, 384)
Retrieved 3 documents (after filtering)
<class 'dict'>
{'source': 'C:\\Users\\kisho\\OneDrive\\Desktop\\AIML LEARN\\data\\pdf\\DEEP LEARNING ( R20A06610 ).pdf', 'creator': 'Microsoft® Word for Microsoft 365', 'page': 3, 'doc_index': 8, 'page_label': '4', 'source_file': 'DEEP LEARNING ( R20A06610 ).pdf', 'content_length': 326, 'file_type': 'pdf', 'producer': 'Microsoft® Word for Microsoft 365', 'total_pages': 170, 'moddate': '2024-08-09T15:33:26+05:30', 'creationdate': '2024-08-09T10:02:17+00:00'}


ENHANCED RAG PIPELINES

In [72]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(
    query,
    retriever,
    llm,
    top_k=5,
    min_score=0.2,
    return_context=False
):
    """
    RAG pipeline with extra features:
    - Returns answer
    - Returns sources
    - Returns confidence score
    - Optionally returns full context
    """

    # Retrieve relevant documents
    results = retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=min_score
    )

    # If no relevant documents are found
    if not results:
        return {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }

    # Prepare context
    context = "\n\n".join(
        [doc["content"] for doc in results]
    )

    # Prepare sources
    sources = []

    for doc in results:

        source_info = {
            "source": doc["metadata"].get(
                "source_file",
                doc["metadata"].get("source", "unknown")
            ),

            "page": doc["metadata"].get(
                "page",
                "unknown"
            ),

            "score": doc["similarity_score"],

            "preview": doc["content"][:300] + "..."
        }

        sources.append(source_info)

    # Calculate confidence
    confidence = max(
        [doc["similarity_score"] for doc in results]
    )

    # Create prompt
    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    # Generate answer using Gemini
    response = llm.invoke(prompt)

    # Prepare output
    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }

    # Include context only if requested
    if return_context:
        output["context"] = context

    return output


# Example usage
result = rag_advanced(
    "Types of Neural Networks",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence"])
print("Context Preview:", result["context"][:300])

Retrieving Documents for query: 'Types of Neural Networks'
Top k: 3, Score threshold: 0.1
Generating embeddings for1texts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Genrated embeddings with shape:(1, 384)
Retrieved 3 documents (after filtering)
Answer: Types of Neural Networks include:

1. Feedforward Neural Networks (FNNs): Simplest type of ANN where data flows in one direction from input to output, used for basic tasks like classification.
2. Convolutional Neural Networks (CNNs): Specialized for processing grid-like data such as images, using convolutional layers to detect spatial hierarchies.
3. Recurrent Neural Networks (RNNs): Process sequential data like time series and natural language, with loops to retain information over time, including variants like LSTMs and GRUs.
4. Generative Adversarial Networks (GANs): Consist of a generator and a discriminator that compete to create realistic data, used for image generation, style transfer, and data augmentation.
Sources: [{'source': 'DL UNIT 1 notes_Updated (1).pdf', 'page': 17, 'score': 0.6117057204246521, 'preview': 'Types of neural networks \n1. Feedforward neural networks (FNNs): They are the

ADVANCED RAG PIPELINE WITH LLM

In [74]:
from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:

        # Retrieve relevant documents
        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""

        else:
            # Combine document contents
            context = "\n\n".join(
                [doc["content"] for doc in results]
            )

            # Get source information
            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file",
                        doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get(
                        "page",
                        "unknown"
                    ),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:120] + "..."
                }
                for doc in results
            ]

            # Create prompt
            prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{question}

Answer:
"""

            # Simulated streaming
            if stream:
                print("Streaming answer:")

                for i in range(0, len(prompt), 80):
                    print(
                        prompt[i:i + 80],
                        end="",
                        flush=True
                    )
                    time.sleep(0.05)

                print()

            # Ask Gemini
            response = self.llm.invoke(prompt)

            # Get answer
            answer = response.content

            # Make sure answer is a string
            if isinstance(answer, list):
                answer = " ".join(
                    str(item) for item in answer
                )

            answer = str(answer)

        # Create citations
        citations = [
            f"[{i + 1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]

        # Add citations
        if citations:
            answer_with_citations = (
                answer
                + "\n\nCitations:\n"
                + "\n".join(citations)
            )
        else:
            answer_with_citations = answer

        # Optional summary
        summary = None

        if summarize and answer:

            summary_prompt = f"""
Summarize the following answer in 2 sentences:

{answer}
"""

            summary_response = self.llm.invoke(
                summary_prompt
            )

            summary = summary_response.content

            # Make summary a string too
            if isinstance(summary, list):
                summary = " ".join(
                    str(item) for item in summary
                )

            summary = str(summary)

        # Store history
        self.history.append({
            "question": question,
            "answer": answer,
            "sources": sources,
            "summary": summary
        })

        # Return result
        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }


# Create Advanced RAG
adv_rag = AdvancedRAGPipeline(
    rag_retriever,
    llm
)

# Run query
result = adv_rag.query(
    "What is Deep Learning?",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)

print("\nFinal Answer:")
print(result["answer"])

print("\nSummary:")
print(result["summary"])

print("\nHistory:")
print(result["history"][-1])

Retrieving Documents for query: 'What is Deep Learning?'
Top k: 3, Score threshold: 0.1
Generating embeddings for1texts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Genrated embeddings with shape:(1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:

Use the following context to answer the question concisely.

Context:
In summary, Deep Learning is a subfield of Machine Learning that involves the useof 
deep neural networks to model and solve complex problems. Deep Learning 
hasachievedsignificantsuccessinvariousfields,anditsuseisexpectedtocontinuetogrow as more 
data becomes available, and more powerful computing resources becomeavailable.

in rec ent years due to the advances in processing power and the availability oflarge 
datasets. Because it is based on artificial neural networks (ANNs) also known as deep neural 
networks (DNNs). These neural networks are inspired by the structure and 
functionofthehumanbrain’s biologicalneurons,andtheyaredesignedtolearnfromlarge 
amounts of data. 
1. Deep Learning is a subfield of Machine Learning that involves the use of neural 
networks to model and solve complex problems. Neural networks are 